# **Семинар 4.** Распознавание именованных сущностей

В прошлый раз мы познакомились с архитектурой RNN и на примере моделирования языка разобрали, как устроен общий пайплайн работы с текстовыми данными.

Сегодня применим этот пайплайн к другой задаче — распознаванию именованных сущностей (англ. Named Entity Recognition, NER) в тексте. Рассмотрим пример:

![ner-01-entities](images/ner-01-entities.png)

В данном случае Alice — имя, Google DeepMind — название организации, а London — место.

Похожим образом можно решать и другие задачи: определять части речи слов в тексте или извлекать данные для заполнения заранее заданных полей (например, адреса, номера телефонов и паспортные данные клиентов) — такая задача называется заполнением слотов (англ. slot filling).

А если вспомнить, что нуклеотидные и аминокислотные последовательности тоже в некотором смысле являются текстами, то размечать можно и их — например, выделять в белках функциональные домены!

![ner-03-protein](images/ner-03-protein.png)

Теперь давайте подумаем, к какой известной нам задаче можно свести NER. Для каждого токена нам нужно решить, принадлежит ли он какой-то сущности или нет. То есть относится ли он к одному из заранее заданных классов. Вывод:

<div style="border-left: 4px solid dodgerblue; padding: 12px 12px 12px 24px; margin: 12px 0; color: dodgerblue; display: flex; align-items: center;">
    <strong>Распознавание именованных сущностей можно сформулировать как задачу многоклассовой классификации для каждого токена.</strong>
</div>

Но ведь классификация — это разновидность обучения с учителем, а значит, нам нужна разметка!

## **4.1. BIO-разметка**

Для NER часто используют BIO-разметку:

- **B** — *beginning*, токен начала сущности;
- **I** — *inside*, токен продолжения той же сущности;
- **O** — *outside*, токен вне сущностей.

Например, `B-Organization` означает начало сущности типа «организация», а `I-Organization` — ее продолжение. Наш пример будет выглядеть следующим образом:

![ner-04-bio](images/ner-04-bio.png)

## **4.2. Собираем LEGO**

Теперь вспомним пайплайн решения задачи моделирования языка с прошлого семинара. Чтобы адаптировать его к NER, достаточно изменить выходную проекцию: теперь скрытые состояния будут проецироваться на набор BIO-классов вместо словаря токенов.

![ner-02-pipeline](images/ner-02-pipeline.png)

## **4.3. ...**

Возьмем [MIT Restaurant Corpus](https://sls.csail.mit.edu/downloads/restaurant/) — небольшой датасет английских запросов о ресторанах. В нем размечены восемь типов фрагментов: название ресторана, кухня, блюдо, местоположение, цена, часы работы, оценка и дополнительные характеристики.

Для удобства воспользуемся [версией проекта TNER](https://huggingface.co/datasets/tner/mit_restaurant). Тексты в ней уже разбиты на токены, а BIO-метки представлены целыми числами. Кроме того, есть готовое разделение на обучающую, валидационную и тестовую подвыборки.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("data/mit_restaurant")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://huggingface.co/datasets/tner/mit_restaurant/resolve/main/dataset"

for filename in ("train.json", "valid.json", "test.json", "label.json"):
    local_path = DATA_DIR / filename

    if not local_path.exists():
        urlretrieve(f"{BASE_URL}/{filename}", local_path)

In [ ]:
import pandas as pd

train = pd.read_json(DATA_DIR / "train.json", lines=True)
valid = pd.read_json(DATA_DIR / "valid.json", lines=True)
test = pd.read_json(DATA_DIR / "test.json", lines=True)

pd.Series({"train": len(train), "valid": len(valid), "test": len(test)}, name="size")

In [ ]:
train.head()

С числами работать удобно, но пока совершенно непонятно, что они означают. Загрузим словарь меток и восстановим разметку одного запроса.

In [ ]:
import json

with open(DATA_DIR / "label.json") as file:
    label2idx = json.load(file)

idx2label = {idx: label for label, idx in label2idx.items()}

pd.Series(label2idx, name="idx")

In [ ]:
example = train.iloc[3]

pd.DataFrame(
    {
        "token": example["tokens"],
        "label": [idx2label[idx] for idx in example["tags"]],
    }
)

В прошлый раз наш словарь состоял из отдельных символов. Сегодня токенами будут слова, поэтому соберем словарь по обучающей подвыборке.

Добавим два специальных токена:

- `[PAD]` — для дополнения коротких последовательностей;
- `[UNK]` — *unknown*, для слов, которых не было в обучающей подвыборке.

Раньше при построении Bag-of-Words мы просто игнорировали OOV-токены. Здесь так делать нельзя: если удалить слово, нарушится соответствие между токенами и их метками. Поэтому незнакомое слово заменим на `[UNK]`, сохранив его место в последовательности.

In [ ]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]"]

VOCABULARY = SPECIAL_TOKENS + sorted({token for tokens in train["tokens"] for token in tokens})

word2idx = {word: idx for idx, word in enumerate(VOCABULARY)}

len(VOCABULARY)

Всего 3 633 токена вместе с `[PAD]` и `[UNK]` — вполне небольшой словарь. Создадим вспомогательную функцию `encode`.

In [ ]:
def encode(tokens: list[str]) -> list[int]:
    return [word2idx.get(token, word2idx["[UNK]"]) for token in tokens]


encode(["a", "cheap", "italian", "restaurant", "abracadabra"])

Теперь напишем класс датасета. Каждый пример будет состоять из двух последовательностей одинаковой длины: индексов токенов и правильных меток.

In [ ]:
import torch
from torch.utils.data import Dataset


class RestaurantDataset(Dataset):
    def __init__(self, data: pd.DataFrame) -> None:
        super().__init__()

        self.inputs = [torch.tensor(encode(tokens)) for tokens in data["tokens"]]
        self.labels = [torch.tensor(tags) for tags in data["tags"]]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.inputs[idx], self.labels[idx]

    def __len__(self) -> int:
        return len(self.inputs)


train_ds = RestaurantDataset(train)
valid_ds = RestaurantDataset(valid)
test_ds = RestaurantDataset(test)

train_ds[0]

Как и в прошлый раз, тексты имеют разную длину. Дополнять придется уже не один тензор, а два: входные токены и правильные метки.

Для входов используем индекс токена `[PAD]`. А для меток возьмем значение `-100`: при вычислении лосса мы будем игнорировать такие позиции.

Почему именно `-100`? Это просто соглашение: у `torch.nn.CrossEntropyLoss` параметр `ignore_index` по умолчанию равен `-100`. При желании можно выбрать другое значение.

Почему просто не дополнить метки нулями? В нашем словаре ноль соответствует метке `O`. Это полноценный класс, который модель должна учиться предсказывать для настоящих токенов. Если игнорировать нули, мы исключим из обучения и такие токены.

In [ ]:
from torch.nn.utils.rnn import pad_sequence


def collate(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[torch.Tensor, torch.Tensor]:
    inputs, labels = zip(*batch, strict=True)

    inputs = pad_sequence(inputs, batch_first=True, padding_value=word2idx["[PAD]"])
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return inputs, labels


inputs, labels = collate([train_ds[i] for i in range(4)])
inputs.shape, labels.shape

In [ ]:
inputs

In [ ]:
labels

Осталось обернуть наши датасеты в `DataLoader`. Перемешивать примеры будем только в обучающей подвыборке.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)
valid_dataloader = DataLoader(valid_ds, batch_size=64, collate_fn=collate)
test_dataloader = DataLoader(test_ds, batch_size=64, collate_fn=collate)

В прошлый раз мы использовали `nn.RNNCell`, которая выполняет один шаг обработки последовательности. Проход по всем позициям пришлось писать самостоятельно.

Теперь воспользуемся модулем [`nn.RNN`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html). Он получает сразу всю последовательность и возвращает скрытые состояния для каждой позиции. При `batch_first=True` вход имеет форму $(B, L, D_{\text{input}})$.

Рассмотрим небольшой пример.

In [ ]:
from torch import nn

B, L, D_input, D_hidden = 4, 5, 10, 20

rnn = nn.RNN(D_input, D_hidden, batch_first=True)
x = torch.rand(B, L, D_input)

output, h_n = rnn(x)

output.shape, h_n.shape

Здесь `output` содержит скрытые состояния **последнего слоя** для всех позиций и для нашей однонаправленной RNN имеет форму $(B, L, D_{\text{hidden}})$. А `h_n` содержит последнее скрытое состояние **каждого слоя и направления**. Например, для нашей однослойной однонаправленной RNN его форма — $(1, B, D_{\text{hidden}})$.

Начальное скрытое состояние можно передать вторым аргументом. Если этого не сделать, `nn.RNN` сама инициализирует его нулями.

Однонаправленная модель читает текст слева направо, поэтому скрытое состояние текущей позиции зависит только от текущего токена и его левого контекста. Для генерации это было необходимо: будущие токены нам неизвестны. Но в случае классификации токенов весь текст уже дан. Почему бы не прочитать текст и справа налево?

Возьмем две RNN: одна будет читать последовательность слева направо, а другая — в обратном направлении. Для каждой позиции объединим их скрытые состояния. Такая архитектура называется **двунаправленной RNN** (англ. *bidirectional RNN*).

$$
h_t = [\overrightarrow{h_t};\overleftarrow{h_t}].
$$

Теперь при размере скрытого состояния $D_{\text{hidden}}$ у каждого направления итоговый вектор имеет размер $2 \times D_{\text{hidden}}$. Поэтому входная размерность выходной головы увеличивается вдвое. В PyTorch для создания двунаправленной RNN достаточно указать `bidirectional=True`.

In [ ]:
K = 3

rnn = nn.RNN(D_input, D_hidden, num_layers=K, batch_first=True, bidirectional=True)
x = torch.rand(B, L, D_input)

output, h_n = rnn(x)

output.shape, h_n.shape

Обратите внимание, что `h_n` теперь имеет форму $(2 \times 3, B, D_{\text{hidden}})$, поскольку у нашей RNN три слоя и два направления.

Но есть и другая важная деталь. Мы дополнили короткие последовательности токенами `[PAD]`, но **RNN не знает, что это служебные позиции**. Если дополнять последовательности справа, то при чтении:

- слева направо токены `[PAD]` не попадут в контекст настоящих токенов;
- справа налево возникнет проблема: модель сначала обработает `[PAD]`.

Как следствие, скрытые состояния настоящих токенов будут зависеть от токенов `[PAD]`, ведь даже нулевой входной вектор не гарантирует, что скрытое состояние останется неизменным.

Чтобы решить эту проблему, в PyTorch используют **упакованные последовательности** (англ. *packed sequences*). Функция `pack_padded_sequence` принимает дополненный батч и настоящие длины текстов. Используя эту информацию, **RNN обрабатывает только настоящие токены**, а `pad_packed_sequence` преобразует результат обратно.

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

x = torch.tensor(
    [
        [1, 2, 0],
        [3, 4, 5],
        [6, 0, 0],
    ]
)
lengths = torch.tensor([2, 3, 1])

packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)

print(type(packed))
print(packed.data)
print(packed.batch_sizes)

In [ ]:
pad_packed_sequence(packed, batch_first=True)

In [ ]:
from lightning import LightningModule
from lightning.pytorch.utilities.types import OptimizerLRScheduler


class RNNForTokenClassification(LightningModule):
    """Рекуррентная нейронная сеть для классификации токенов.

    Условные обозначения:
        B - размер батча;
        L - длина последовательности;
        D_input - размер входного эмбеддинга;
        D_hidden - размер скрытого состояния;
        D_output - размер выхода RNN;
        C - количество классов.
    """

    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        input_dim: int = 64,
        hidden_dim: int = 64,
        bidirectional: bool = False,
        pad_token_id: int = 0,
    ) -> None:
        super().__init__()

        self.vocab_size = vocab_size
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id

        self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=pad_token_id)
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)

        output_dim = hidden_dim * (2 if bidirectional else 1)

        self.head = nn.Linear(output_dim, num_classes)

        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # считаем настоящие длины последовательностей
        lengths = (inputs != self.pad_token_id).sum(dim=1).cpu()  # (B,)

        inputs_embeds = self.embedding(inputs)  # (B, L, D)

        # упаковываем `inputs_embeds`
        packed_inputs_embeds = pack_padded_sequence(inputs_embeds, lengths, batch_first=True, enforce_sorted=False)

        packed_hidden_states, _ = self.rnn(packed_inputs_embeds)

        # распаковываем `hidden_states`
        hidden_states, _ = pad_packed_sequence(packed_hidden_states, batch_first=True)

        return self.head(hidden_states)  # (B, L, C)

    def training_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int | None = None) -> torch.Tensor:
        inputs, labels = batch

        logits = self(inputs)
        loss = self.loss_fn(logits.reshape(-1, self.num_classes), labels.reshape(-1))

        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=(labels != -100).sum().item(),
        )
        return loss

    def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int | None = None) -> None:
        inputs, labels = batch

        logits = self(inputs)
        loss = self.loss_fn(logits.reshape(-1, self.num_classes), labels.reshape(-1))

        self.log(
            "val_loss",
            loss,
            prog_bar=True,
            batch_size=(labels != -100).sum().item(),
        )

    def configure_optimizers(self) -> OptimizerLRScheduler:
        return torch.optim.AdamW(self.parameters(), lr=2.5e-3)

In [ ]:
model = RNNForTokenClassification(vocab_size=len(VOCABULARY), num_classes=len(label2idx))
model(inputs).shape

Модель готова — можно запускать обучение. Валидационная подвыборка поможет следить за качеством во время обучения. Тестовую пока оставим в стороне.

In [ ]:
from lightning import Trainer

trainer = Trainer(
    max_epochs=10,
    gradient_clip_val=1.0,
    logger=False,
    enable_checkpointing=False,
)
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=valid_dataloader)

Модель обучена. Как понять, насколько хорошо она справляется с задачей? Можно посчитать accuracy по токенам, но у этой метрики есть проблема: значительная часть токенов имеет метку `O`.

Посчитаем, какую accuracy получила бы модель, всегда предсказывающая `O`.

In [ ]:
train_labels = [label for tags in train["tags"] for label in tags]

sum(label == label2idx["O"] for label in train_labels) / len(train_labels)

Но такая модель не нашла бы ни одной сущности. Кроме того, нам важно правильно определить весь фрагмент, включая его границы. Например, если вместо `central park` модель выделила только `central`, сущность распознана неверно. При строгой оценке правильным считается только полное совпадение границ и типа.

Будем считать F1 по целым сущностям. Для этого воспользуемся библиотекой [`seqeval`](https://github.com/chakki-works/seqeval). Рассмотрим два варианта предсказания.

In [ ]:
from seqeval.metrics import f1_score
from seqeval.scheme import IOB2

gold_tags = [["B-Location", "I-Location"]]

In [ ]:
pred_tags = [["B-Location", "O"]]

f1_score(gold_tags, [["B-Location", "O"]], mode="strict", scheme=IOB2)

In [ ]:
pred_tags = [["B-Location", "I-Location"]]

f1_score(gold_tags, pred_tags, mode="strict", scheme=IOB2)

Теперь получим предсказания нашей модели. В простейшем случае будем выбирать для каждого токена метку с максимальным логитом.

In [ ]:
@torch.inference_mode()
def predict(
    model: RNNForTokenClassification,
    dataloader: DataLoader,
    device: torch.device | str = "cpu",
) -> tuple[list[list[str]], list[list[str]]]:
    model = model.eval().to(device)

    all_true_labels, all_pred_labels = [], []

    for inputs, batch_true_labels in dataloader:
        logits = model(inputs.to(device))

        batch_pred_labels = logits.argmax(dim=-1).cpu()

        for pred_labels, true_labels in zip(batch_pred_labels, batch_true_labels, strict=True):
            mask = true_labels != -100

            all_true_labels.append([idx2label[idx] for idx in true_labels[mask].tolist()])
            all_pred_labels.append([idx2label[idx] for idx in pred_labels[mask].tolist()])

    return all_true_labels, all_pred_labels


valid_true, valid_pred = predict(model, valid_dataloader)

f1_score(valid_true, valid_pred, mode="strict", scheme=IOB2)

In [ ]:
bidirectional_model = RNNForTokenClassification(
    vocab_size=len(VOCABULARY),
    num_classes=len(label2idx),
    bidirectional=True,
)
bidirectional_trainer = Trainer(
    max_epochs=10,
    gradient_clip_val=1.0,
    logger=False,
    enable_checkpointing=False,
)
bidirectional_trainer.fit(
    bidirectional_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=valid_dataloader,
)

In [ ]:
valid_true_bidirectional, valid_pred_bidirectional = predict(bidirectional_model, valid_dataloader)

pd.DataFrame(
    {
        "model": ["RNN", "Bidirectional RNN"],
        "valid F1": [
            f1_score(valid_true, valid_pred, mode="strict", scheme=IOB2),
            f1_score(valid_true_bidirectional, valid_pred_bidirectional, mode="strict", scheme=IOB2),
        ],
    }
).set_index("model").round(3)

Теперь оценим лучшую модель на тестовой выборке.

In [ ]:
test_true_bidirectional, test_pred_bidirectional = predict(bidirectional_model, test_dataloader)

f1_score(test_true_bidirectional, test_pred_bidirectional, mode="strict", scheme=IOB2)

In [ ]:
from seqeval.metrics import classification_report

print(
    classification_report(
        test_true_bidirectional,
        test_pred_bidirectional,
        mode="strict",
        scheme=IOB2,
        digits=3,
        zero_division=0,
    )
)

Посмотрим на один запрос из тестовой подвыборки и сравним правильные метки с предсказаниями обеих моделей.

In [ ]:
i = 42

pd.DataFrame(
    {
        "token": test.iloc[i]["tokens"],
        "true": test_true_bidirectional[i],
        "pred": test_pred_bidirectional[i],
    }
)

In [ ]:
@torch.inference_mode()
def predict_text(
    model: RNNForTokenClassification,
    text: str,
    device: torch.device | str = "cpu",
) -> pd.DataFrame:
    model = model.eval().to(device)

    tokens = text.lower().split()
    inputs = torch.tensor([encode(tokens)], device=device)

    predictions = model(inputs).argmax(dim=-1)[0].tolist()

    return pd.DataFrame({"token": tokens, "label": [idx2label[idx] for idx in predictions]})


predict_text(bidirectional_model, "find a cheap italian restaurant near central park")

Для чуть более удобного просмотра подсветим найденные сущности.

In [ ]:
from html import escape

from IPython.display import HTML, display
from seqeval.metrics.sequence_labeling import get_entities

ENTITY_COLORS = {
    "Rating": "#fef3c7",
    "Amenity": "#e2e8f0",
    "Location": "#dbeafe",
    "Restaurant_Name": "#fce7f3",
    "Price": "#dcfce7",
    "Hours": "#ede9fe",
    "Dish": "#ffedd5",
    "Cuisine": "#cffafe",
}


def highlight_entities(prediction: pd.DataFrame) -> HTML:
    tokens = prediction["token"].tolist()
    labels = prediction["label"].tolist()
    parts, cursor = [], 0

    for entity_type, start, end in get_entities(labels):
        parts.extend(escape(token) for token in tokens[cursor:start])
        text = escape(" ".join(tokens[start : end + 1]))
        color = ENTITY_COLORS[entity_type]
        parts.append(
            f'<span style="background: {color}; color: #172033; padding: 4px 7px; '
            f'border-radius: 5px;">{text} '
            f'<small style="color: #475569;">[{entity_type}]</small></span>'
        )
        cursor = end + 1

    parts.extend(escape(token) for token in tokens[cursor:])
    return HTML('<div style="line-height: 2.4; margin: 12px 0;">' + " ".join(parts) + "</div>")

In [ ]:
queries = [
    "find a cheap italian restaurant near central park",
    "is there a chinese restaurant in boston open tonight",
    "i want sushi near harvard square",
]

for query in queries:
    prediction = predict_text(bidirectional_model, query)
    display(highlight_entities(prediction))